# Native Y-half cube (S-gate cap)

A Y-basis logical measurement, compiled natively through `tqec` to a
`stim.Circuit`.

The first graph is a `ZXZ` memory cube capped by a `Y_HALF_CUBE` through a
temporal pipe. The Y cube lowers to a sequence of per-round raw layers --- the
transition round that folds the patch onto its Y boundary, `k` boundary
(padding) rounds, and a transversal final measurement --- reproducing Craig
Gidney's [in-place Y-basis
measurement](https://quantum-journal.org/papers/q-2024-04-08-1310/). The memory
round the transition closes its seam detectors against is the temporal pipe's
own junction round. No fictitious `MPP` elements are used.

A **lone** Y column has no closed correlation surface, so its logical-Y readout
is physically random and no `OBSERVABLE_INCLUDE` is emitted --- every detector
is still deterministic.

In [ ]:
from tqec import BlockGraph, compile_block_graph
from tqec.computation.cube import ZXCube, LeafCubeKind
from tqec.utils.position import Position3D

g = BlockGraph("y_capped_column")
g.add_cube(Position3D(0, 0, 0), ZXCube.from_str("ZXZ"))   # memory cube
g.add_cube(Position3D(0, 0, 1), LeafCubeKind.Y_HALF_CUBE)  # Y-basis measurement cap
g.add_pipe(Position3D(0, 0, 0), Position3D(0, 0, 1))       # temporal pipe
g.view_as_html()

In [ ]:
k = 1  # code distance d = 2k + 1 = 3
circuit = compile_block_graph(g, observables="auto").generate_stim_circuit(k=k)

print(f"qubits:     {circuit.num_qubits}")
print(f"ticks:      {circuit.num_ticks}")
print(f"detectors:  {circuit.num_detectors}")
print(f"observables:{circuit.num_observables}  (lone Y column → random readout, none emitted)")
print(f"uses MPP:   {any(inst.name == 'MPP' for inst in circuit.flattened())}")

In [ ]:
# Every detector is deterministic: the detector error model builds without error.
dem = circuit.detector_error_model(decompose_errors=False)
print("detector error model built — all detectors deterministic")

## Detector-slice diagram

Each coloured region is a detector (a set of measurements whose parity is
deterministic). The seam detectors straddle the transition round, linking the
junction round below to the folded Y patch. Scroll horizontally through the
rounds.

In [ ]:
circuit.diagram("detslice-with-ops-svg")

## Two Y caps: the $S^2 = Z$ observable

The interesting case is a Y cap that runs *alongside* a computation that keeps
going, which is what an `S` gate looks like: a side branch is measured out in
the Y basis while the main column continues. The cap is `k+3` rounds and a
continuing memory cube is `2k+1`, so the two are merged with mismatched temporal
schedules --- the cap is start-aligned, and whichever block is shorter either
drops out (the cap, which measures all its data out) or is padded (the column,
which must carry its logical state onwards).

Two such gadgets close a correlation surface: `in . out . Y . Y`, the
$S^2 = Z$ algebra. `find_correlation_surfaces` finds it, and each cap
contributes its transition-round logical-Y readout to the observable.

In [ ]:
from typing import Literal

def build_yy(memory_kind: Literal["ZXZ", "XZX"]):
    g = BlockGraph("yy")
    
    b0 = Position3D(0, 0, 0)
    b1 = Position3D(0, 0, 1)
    b2 = Position3D(0, 0, 2)
    b3 = Position3D(0, 0, 3)
    b4 = Position3D(0, 0, 4)
    c1 = Position3D(1, 0, 1)
    c3 = Position3D(1, 0, 3)
    y2 = Position3D(1, 0, 2)
    y4 = Position3D(1, 0, 4)
    
    g.add_cube(b0, memory_kind)
    g.add_cube(b1, memory_kind)
    g.add_cube(b2, memory_kind)
    g.add_cube(b3, memory_kind)
    g.add_cube(b4, memory_kind)
    g.add_cube(c1, memory_kind)
    g.add_cube(c3, memory_kind)
    g.add_cube(y2, LeafCubeKind.Y_HALF_CUBE)
    g.add_cube(y4, LeafCubeKind.Y_HALF_CUBE)
    
    g.add_pipe(b0, b1)
    g.add_pipe(b1, b2)
    g.add_pipe(b2, b3)
    g.add_pipe(b3, b4)
    g.add_pipe(b1, c1)
    g.add_pipe(b3, c3)
    g.add_pipe(c1, y2)
    g.add_pipe(c3, y4)

    return g

g_yy = build_yy("XZX")
obs = g_yy.find_correlation_surfaces()
print(f"{len(obs)} observable(s) found")

g_yy.view_as_html(pop_faces_at_directions=("-Y",), show_correlation_surface=obs[0])

In [ ]:
circuit_yy = compile_block_graph(g_yy, observables=obs).generate_stim_circuit(k=2)

print(f"observables: {circuit_yy.num_observables}")
print(f"surface:     {obs[0].external_stabilizer_on_graph(g_yy)}")
# Builds only if every detector *and* the observable is deterministic.
circuit_yy.detector_error_model(decompose_errors=False)

_, samples = circuit_yy.compile_detector_sampler().sample(200, separate_observables=True)
print(f"observable values over 200 shots: {set(bool(v) for v in samples.reshape(-1))}")

The observable takes the same value on every shot: the surface is closed.

The last diagram shows the two caps side by side with the continuing column.

In [ ]:
circuit_yy.diagram("detslice-with-ops-svg")